In [ ]:
#@title Setup — run once { display-mode: "form" }
import subprocess, threading, time

subprocess.run(["pip", "install", "uv", "-q"], check=True)
subprocess.run(
    ["uv", "pip", "install", "--system", "-q",
     "git+https://github.com/bruno-ritmista/samba@c8eaa29da242271851978d553fe6a5bdf7ae2e1b#subdirectory=sheets_to_banana"],
    check=True
)

def _keep_alive():
    while True:
        time.sleep(60)

_t = threading.Thread(target=_keep_alive, daemon=True)
_t.start()

print("\u2705 Setup complete \u2014 scroll down and fill in the form below.")

In [ ]:
#@title Generate BananaDrum link { run: "auto" }

sheets_url = "" #@param {type:"string", placeholder:"Paste the Google Sheets link here"}
break_number = 1 #@param {type:"integer"}
tempo = 120 #@param {type:"integer"}

import re

def _run():
    val = sheets_url.strip()

    if not val:
        print("Paste a Google Sheets link in the field above, then run this cell.")
        return

    if re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', val):
        full_url = val
    elif re.fullmatch(r'[a-zA-Z0-9_-]+', val):
        full_url = f"https://docs.google.com/spreadsheets/d/{val}"
    else:
        print("\u274c That doesn't look like a Google Sheets link. "
              "Please copy the link from the sheet and try again.")
        return

    try:
        from sheets_to_banana.fetch import fetch_csv
        from sheets_to_banana.parse import parse_sheet
        from sheets_to_banana.mapping import map_break
        from sheets_to_banana.encode import encode_url
    except ImportError:
        print("\u274c Setup not complete. Please run the Setup cell first.")
        return

    try:
        csv_text = fetch_csv(full_url)
    except Exception:
        print("\u274c Could not read your Google Sheet. "
              "Please check the link and try again.")
        return

    breaks = parse_sheet(csv_text)
    if not breaks:
        print("\u274c No breaks found in the sheet. "
              "Please check that the sheet has the expected format.")
        return

    if break_number < 1 or break_number > len(breaks):
        print(f"\u274c Break {break_number} not found. "
              f"This sheet has {len(breaks)} break(s): 1\u2013{len(breaks)}.")
        return

    brk = breaks[break_number - 1]
    tracks = map_break(brk)
    if not tracks:
        print(f"\u274c Break {break_number} \"{brk.name}\" has no recognised instruments.")
        return

    n_bars = max(len(t.notes) for t in tracks) // 16
    url = encode_url(tracks, tempo=tempo, n_bars=n_bars)

    print(f"Break {break_number}: {brk.name}")
    print(f"{n_bars} bars \u2014 {len(tracks)} instruments")
    print()
    print("\U0001f517 " + url)

_run()